# Phase 8 — Détection d'anomalies IoT par Machine Learning

**PFE — Sécurisation des flux de communication pour un écosystème IoT simulé**  
**FST / Tunisie Telecom** | Amine Ghannouchi

---

## Objectifs
1. **EDA** — Exploration et compréhension du dataset
2. **Feature Engineering** — Sélection et préparation des features
3. **IsolationForest** — Détection d'anomalies non-supervisée
4. **RandomForest** — Classification supervisée (normal vs attaque)
5. **Évaluation** — Confusion matrix, ROC-AUC, F1-score
6. **Visualisations** — Pour le rapport PFE

## Dataset
- **76 000 lignes**, 37 colonnes
- **7 types d'attaques** : dos, replay, mitm, spoofing, tls_downgrade, data_exfiltration, rogue_gateway
- Protocoles : MQTT, CoAP, HTTP, AMQP
- Cible : `label_secure` (1=normal, 0=attaque)

## 0 — Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, f1_score, precision_score, recall_score
)
from sklearn.pipeline import Pipeline
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Chemins
DATASET = '../../datasets/iot_communication_security_dataset_7 (1).csv'
RESULTS = '../../results/analysis/ml/'
MODELS  = '../models/'
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(MODELS,  exist_ok=True)

print('Imports OK')
print(f'Results dir : {os.path.abspath(RESULTS)}')

## 1 — Chargement et exploration (EDA)

In [ ]:
df = pd.read_csv(DATASET)
print(f'Shape  : {df.shape}')
print(f'Colonnes : {df.shape[1]}')
df.head(3)

In [ ]:
print('=== Types de données ===')
print(df.dtypes.value_counts())
print()
print('=== Valeurs manquantes ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'Aucune valeur manquante ✅')

In [ ]:
# Distribution des classes
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) label_secure
counts = df['label_secure'].value_counts()
labels = ['Normal (1)', 'Attaque (0)']
axes[0].pie(counts, labels=labels, autopct='%1.1f%%',
            colors=['#2196F3','#F44336'], startangle=90)
axes[0].set_title('Distribution label_secure')

# 2) attack_type
at = df['attack_type'].value_counts()
at.plot(kind='barh', ax=axes[1], color='#FF7043')
axes[1].set_title('Types d\'attaques')
axes[1].set_xlabel('Nombre de lignes')

# 3) protocol
df['protocol'].value_counts().plot(kind='bar', ax=axes[2], color='#42A5F5', rot=0)
axes[2].set_title('Distribution protocoles')
axes[2].set_ylabel('Nombre de lignes')

plt.tight_layout()
plt.savefig(RESULTS + 'eda_classes_distribution.png', bbox_inches='tight')
plt.show()
print(f'Sauvegardé : {RESULTS}eda_classes_distribution.png')

In [ ]:
# Statistiques descriptives des features numériques clés
num_features = ['rtt_ms','jitter_ms','packet_loss_pct','retransmission_count',
                'cpu_usage_pct','mem_usage_pct','failed_auth_1h',
                'conn_attempts_5m','payload_entropy','risk_score',
                'duplicate_msg_rate_pct','anomaly_score_baseline']

df[num_features + ['label_secure']].groupby('label_secure').mean().T.style.background_gradient(cmap='RdYlGn_r')

In [ ]:
# Heatmap de corrélation
corr = df[num_features].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Matrice de corrélation — Features numériques IoT', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS + 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots : features clés par attack_type
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
features_box = ['rtt_ms','failed_auth_1h','conn_attempts_5m',
                'duplicate_msg_rate_pct','payload_entropy','risk_score']

for ax, feat in zip(axes.flatten(), features_box):
    order = df.groupby('attack_type')[feat].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x='attack_type', y=feat, order=order,
                ax=ax, palette='Set2', fliersize=2)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.set_title(feat)
    ax.set_xlabel('')

plt.suptitle('Distribution des features clés par type d\'attaque', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS + 'eda_boxplots_by_attack.png', bbox_inches='tight')
plt.show()

## 2 — Feature Engineering

In [ ]:
# Copie de travail
df_ml = df.copy()

# Encodage des variables catégorielles
cat_cols = ['protocol','tls_version','auth_method','cert_status',
            'src_zone','dst_zone','message_type','device_type']

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col].astype(str))
    le_dict[col] = le

# Features sélectionnées pour le modèle
FEATURES = [
    # Réseau
    'payload_bytes','packet_count','session_duration_ms',
    'rtt_ms','jitter_ms','packet_loss_pct','retransmission_count',
    'topic_frequency_1m','duplicate_msg_rate_pct','payload_entropy',
    # Sécurité
    'qos_level','key_rotation_age_days','firmware_age_days',
    'conn_attempts_5m','failed_auth_1h',
    'anomaly_score_baseline','risk_score',
    # Device
    'cpu_usage_pct','mem_usage_pct','signal_rssi_dbm','battery_pct',
    # Encodés
    'protocol_enc','tls_version_enc','auth_method_enc','cert_status_enc',
    'src_zone_enc','dst_zone_enc','message_type_enc','device_type_enc'
]

X = df_ml[FEATURES]
y = df_ml['label_secure']  # 1=normal, 0=attaque
y_attack = df_ml['attack_type']

print(f'Features : {len(FEATURES)}')
print(f'X shape  : {X.shape}')
print(f'y distribution:\n{y.value_counts()}')

## 3 — IsolationForest (détection non-supervisée)

In [ ]:
# IsolationForest entraîné uniquement sur les données normales
# (simule un déploiement réel : on n'a pas les labels d'attaque)

X_normal = X[y == 1]  # Entraînement sur données normales seulement
print(f'Entraînement sur {len(X_normal)} lignes normales')

# contamination = proportion d'anomalies attendue dans le jeu de test
contamination = round(len(y[y==0]) / len(y), 4)
print(f'Contamination estimée : {contamination} ({contamination*100:.1f}%)')

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_normal)

# Prédiction sur tout le dataset
# IsolationForest retourne -1 (anomalie) ou 1 (normal)
iso_preds_raw = iso_forest.predict(X)
# Convertir : -1 → 0 (attaque), 1 → 1 (normal)
iso_preds = np.where(iso_preds_raw == 1, 1, 0)
iso_scores = iso_forest.decision_function(X)  # Score d'anomalie

print(f'\nPrédictions IsolationForest :')
print(f'  Normal  (1) : {(iso_preds==1).sum()}')
print(f'  Attaque (0) : {(iso_preds==0).sum()}')

In [ ]:
# Évaluation IsolationForest
print('=== Rapport IsolationForest (détection non-supervisée) ===')
print(classification_report(y, iso_preds,
      target_names=['Attaque (0)', 'Normal (1)']))

iso_f1  = f1_score(y, iso_preds)
iso_roc = roc_auc_score(y, iso_scores)
print(f'F1-score  : {iso_f1:.4f}')
print(f'ROC-AUC   : {iso_roc:.4f}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y, iso_preds,
    display_labels=['Attaque', 'Normal'],
    cmap='Blues', ax=ax
)
ax.set_title(f'IsolationForest — Matrice de confusion\n(F1={iso_f1:.3f}, ROC-AUC={iso_roc:.3f})')
plt.tight_layout()
plt.savefig(RESULTS + 'iso_forest_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribution des scores d'anomalie IsolationForest
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(iso_scores[y==1], bins=80, alpha=0.6, color='#2196F3', label='Normal', density=True)
ax.hist(iso_scores[y==0], bins=80, alpha=0.6, color='#F44336', label='Attaque', density=True)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='Seuil de décision')
ax.set_title('Distribution des scores d\'anomalie IsolationForest')
ax.set_xlabel('Score d\'anomalie (plus négatif = plus anormal)')
ax.set_ylabel('Densité')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS + 'iso_forest_scores_distribution.png', bbox_inches='tight')
plt.show()

## 4 — RandomForest (classification supervisée)

In [ ]:
# Split train/test stratifié (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]} lignes  ({y_train.value_counts()[0]} attaques, {y_train.value_counts()[1]} normaux)')
print(f'Test  : {X_test.shape[0]} lignes  ({y_test.value_counts()[0]} attaques, {y_test.value_counts()[1]} normaux)')

# RandomForest avec class_weight pour compenser le déséquilibre (88%/12%)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 0]  # Proba de classe "attaque" (0)

print('\nEntraînement terminé ✅')

In [ ]:
# Rapport de classification détaillé
print('=== Rapport RandomForest (classification supervisée) ===')
print(classification_report(y_test, y_pred,
      target_names=['Attaque (0)', 'Normal (1)']))

rf_f1  = f1_score(y_test, y_pred)
rf_roc = roc_auc_score(y_test, 1 - y_proba)  # 1 - proba_attack = proba_normal
print(f'F1-score global : {rf_f1:.4f}')
print(f'ROC-AUC         : {rf_roc:.4f}')

In [ ]:
# Confusion matrix RandomForest
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Matrice de confusion
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Attaque', 'Normal'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'RandomForest — Matrice de confusion\n(F1={rf_f1:.3f})')

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, 1 - y_proba)
axes[1].plot(fpr, tpr, color='#2196F3', lw=2,
             label=f'ROC-AUC = {rf_roc:.3f}')
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Aléatoire')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
axes[1].set_xlabel('Taux de faux positifs')
axes[1].set_ylabel('Taux de vrais positifs')
axes[1].set_title('Courbe ROC — RandomForest')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS + 'rf_confusion_roc.png', bbox_inches='tight')
plt.show()

In [ ]:
# Importance des features
importances = pd.Series(rf.feature_importances_, index=FEATURES)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.sort_values().plot(kind='barh', ax=ax, color='#42A5F5')
ax.set_title('Top 20 features — Importance RandomForest', fontsize=13)
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.savefig(RESULTS + 'rf_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 10 features :')
for feat, imp in top20.head(10).items():
    print(f'  {feat:<35} : {imp:.4f}')

## 5 — Classification multi-classe par type d'attaque

In [ ]:
# Classifier les 7 types d'attaques
le_attack = LabelEncoder()
y_multi = le_attack.fit_transform(y_attack)

X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

rf_multi = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_multi.fit(X_tr_m, y_tr_m)
y_pred_m = rf_multi.predict(X_te_m)

print('=== Rapport multi-classes (7 types d\'attaques) ===')
print(classification_report(y_te_m, y_pred_m,
      target_names=le_attack.classes_))

In [ ]:
# Confusion matrix multi-classes
fig, ax = plt.subplots(figsize=(9, 7))
cm = confusion_matrix(y_te_m, y_pred_m)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_attack.classes_,
            yticklabels=le_attack.classes_, ax=ax)
ax.set_title('Matrice de confusion — Classification multi-classes', fontsize=13)
ax.set_ylabel('Réel')
ax.set_xlabel('Prédit')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(RESULTS + 'rf_multiclass_confusion.png', bbox_inches='tight')
plt.show()

## 6 — Comparaison des modèles

In [ ]:
# Tableau comparatif des métriques
results = pd.DataFrame({
    'Modèle': ['IsolationForest\n(non-supervisé)', 'RandomForest\n(supervisé)'],
    'Précision (attaque)': [
        precision_score(y, iso_preds, pos_label=0),
        precision_score(y_test, y_pred, pos_label=0)
    ],
    'Rappel (attaque)': [
        recall_score(y, iso_preds, pos_label=0),
        recall_score(y_test, y_pred, pos_label=0)
    ],
    'F1-score': [iso_f1, rf_f1],
    'ROC-AUC': [iso_roc, rf_roc]
})

print(results.to_string(index=False))

# Graphique comparatif
fig, ax = plt.subplots(figsize=(9, 4))
metrics = ['Précision (attaque)', 'Rappel (attaque)', 'F1-score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, results.iloc[0][metrics], width,
               label='IsolationForest', color='#FF7043', alpha=0.85)
bars2 = ax.bar(x + width/2, results.iloc[1][metrics], width,
               label='RandomForest', color='#2196F3', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparaison IsolationForest vs RandomForest — Détection d\'anomalies IoT')
ax.legend()
ax.bar_label(bars1, fmt='%.2f', fontsize=9)
ax.bar_label(bars2, fmt='%.2f', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS + 'models_comparison.png', bbox_inches='tight')
plt.show()

## 7 — Sauvegarde des modèles

In [ ]:
joblib.dump(iso_forest, MODELS + 'isolation_forest.pkl')
joblib.dump(rf,         MODELS + 'random_forest_binary.pkl')
joblib.dump(rf_multi,   MODELS + 'random_forest_multiclass.pkl')
joblib.dump(le_dict,    MODELS + 'label_encoders.pkl')
joblib.dump(le_attack,  MODELS + 'label_encoder_attack.pkl')

print('Modèles sauvegardés :')
for f in os.listdir(MODELS):
    size = os.path.getsize(MODELS + f) / 1024
    print(f'  {f:<45} {size:.0f} Ko')

## 8 — Résumé pour le rapport PFE

### Résultats obtenus

| Modèle | Type | F1-score | ROC-AUC | Avantage |
|--------|------|----------|---------|----------|
| IsolationForest | Non-supervisé | ~0.6x | ~0.8x | Pas de labels requis |
| RandomForest | Supervisé | ~0.9x | ~0.9x | Meilleure précision |

### Points à inclure dans le rapport

**Section Rapport — Chapitre 6 :** "Détection d'anomalies par Machine Learning"
- Description du dataset (76 000 lignes, 37 features, 7 types d'attaques)
- Features engineering : encodage catégorielles, 28 features retenues
- Comparaison supervisé vs non-supervisé
- Figures : `eda_classes_distribution.png`, `rf_feature_importance.png`, `models_comparison.png`, `rf_multiclass_confusion.png`

**Annexes :** Ce notebook + `ml/models/`